# Vision Stats: Season 2025 vs 2026 — Bronze I & Emerald I

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, normaltest, mannwhitneyu, ttest_ind

sns.set_theme(style='whitegrid')

FEATURES = [
    'visionScorePerMinute',
    'visionScore',
    'controlWardsPlaced',
    'stealthWardsPlaced',
    'visionWardsBoughtInGame',
    'wardsPlaced',
    'wardsKilled',
]

## Load data

In [ ]:
def extract_participant_features(filepath):
    """Return list of feature dicts — one per participant in the match."""
    with open(filepath, encoding='utf-8') as f:
        data = json.load(f)

    match_info = data.get('match_info', {})
    participants = data['match_data']['info']['participants']
    rows = []
    for p in participants:
        challenges = p.get('challenges', {})
        row = {
            # top-level fields
            'visionScore':            p.get('visionScore', np.nan),
            'visionWardsBoughtInGame':p.get('visionWardsBoughtInGame', np.nan),
            'wardsPlaced':            p.get('wardsPlaced', np.nan),
            'wardsKilled':            p.get('wardsKilled', np.nan),
            # challenge fields
            'visionScorePerMinute':   challenges.get('visionScorePerMinute', np.nan),
            'controlWardsPlaced':     challenges.get('controlWardsPlaced', np.nan),
            'stealthWardsPlaced':     challenges.get('stealthWardsPlaced', np.nan),
        }
        rows.append(row)
    return rows


def load_tier(season, tier):
    pattern = os.path.join('data', season, tier, '*.json')
    files = glob.glob(pattern)
    all_rows = []
    for fp in files:
        try:
            all_rows.extend(extract_participant_features(fp))
        except Exception as e:
            print(f'Skipping {fp}: {e}')
    df = pd.DataFrame(all_rows)
    df['season'] = season
    df['tier'] = tier
    return df


b25 = load_tier('season2025', 'bronze1')
b26 = load_tier('season2026', 'bronze1')
e25 = load_tier('season2025', 'emerald1')
e26 = load_tier('season2026', 'emerald1')

df_all = pd.concat([b25, b26, e25, e26], ignore_index=True)
df_all['group'] = df_all['season'] + '_' + df_all['tier']

print('Rows per group:')
print(df_all['group'].value_counts())

In [ ]:
b25.describe()

In [ ]:
b26.describe()

In [ ]:
e25.describe()

In [ ]:
e26.describe()

## Distributions — histograms

In [ ]:
groups = ['season2025_bronze1', 'season2026_bronze1', 'season2025_emerald1', 'season2026_emerald1']
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange']

fig, axes = plt.subplots(len(FEATURES), 1, figsize=(12, 4 * len(FEATURES)))
for ax, feat in zip(axes, FEATURES):
    for grp, col in zip(groups, colors):
        vals = df_all.loc[df_all['group'] == grp, feat].dropna()
        vals.plot.hist(ax=ax, alpha=0.4, bins=40, color=col, density=True, label=grp)
    ax.set_title(feat)
    ax.legend(fontsize=8)
    ax.set_xlabel(feat)
plt.tight_layout()
plt.savefig('figures/distributions.png', dpi=100)
plt.show()

## Boxplots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, feat in zip(axes.flat, FEATURES):
    sns.boxplot(data=df_all, x='group', y=feat, ax=ax, hue='group', legend=False,
                order=groups, palette=colors)
    ax.set_title(feat)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig('figures/boxplots.png', dpi=100)
plt.show()

## Normality tests

- **Shapiro-Wilk** (best for n < 5000, used when subsample ≤ 5000)
- **D'Agostino–Pearson** (`normaltest`) as a complement

α = 0.05; if both tests fail to reject H₀ (p > 0.05) → assume normal; otherwise → non-normal.

In [ ]:
ALPHA = 0.05
SHAPIRO_MAX = 5500   # shapiro is slow, go for random sample when population is too big

normality_results = []
for grp in groups:
    for feat in FEATURES:
        vals = df_all.loc[df_all['group'] == grp, feat].dropna().values
        n = len(vals)

        # Shapiro-Wilk on subsample if large
        sample = np.random.default_rng(42).choice(vals, size=min(n, SHAPIRO_MAX), replace=False)
        sw_stat, sw_p = shapiro(sample)

        # D'Agostino-Pearson (needs n >= 20)
        if n >= 20:
            dp_stat, dp_p = normaltest(vals)
        else:
            dp_stat, dp_p = np.nan, np.nan

        is_normal = (sw_p > ALPHA) and (dp_p > ALPHA if not np.isnan(dp_p) else True)

        normality_results.append({
            'group': grp, 'feature': feat, 'n': n,
            'shapiro_stat': round(sw_stat, 4), 'shapiro_p': round(sw_p, 6),
            'dagostino_stat': round(dp_stat, 4) if not np.isnan(dp_stat) else np.nan,
            'dagostino_p': round(dp_p, 6) if not np.isnan(dp_p) else np.nan,
            'normal': is_normal,
        })

norm_df = pd.DataFrame(normality_results)
norm_df

## Statistical comparison tests

Comparison pairs:
1. **Bronze I**: season2025 vs season2026
2. **Emerald I**: season2025 vs season2026

Decision rule:
- If **both** groups are normal → **Welch's t-test** (accounts for unequal variances via Levene's test)
- Otherwise → **Mann-Whitney U test** (non-parametric)

Effect size:
- t-test → **Cohen's d**
- Mann-Whitney → **rank-biserial correlation r**

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0.0

def rank_biserial(u_stat, n1, n2):
    return 1 - (2 * u_stat) / (n1 * n2)

def larger_population(a, b, test_name):
    if test_name == "Welch's t-test":
        return '2025' if np.mean(a) > np.mean(b) else '2026'
    else:
        return '2025' if np.median(a) > np.median(b) else '2026'

def is_normal_group(norm_df, grp, feat):
    row = norm_df[(norm_df['group'] == grp) & (norm_df['feature'] == feat)]
    if row.empty:
        return False
    return bool(row.iloc[0]['normal'])


comparison_pairs = [
    ('season2025_bronze1',  'season2026_bronze1',  'Bronze I'),
    ('season2025_emerald1', 'season2026_emerald1', 'Emerald I'),
]

test_results = []
for g1, g2, label in comparison_pairs:
    for feat in FEATURES:
        a = df_all.loc[df_all['group'] == g1, feat].dropna().values
        b = df_all.loc[df_all['group'] == g2, feat].dropna().values

        both_normal = is_normal_group(norm_df, g1, feat) and is_normal_group(norm_df, g2, feat)

        if both_normal:
            stat, p = ttest_ind(a, b, equal_var=False)
            test_name = "Welch's t-test"
            effect = cohens_d(a, b)
            effect_name = "Cohen's d"
        else:
            result = mannwhitneyu(a, b, alternative='two-sided')
            stat, p = result.statistic, result.pvalue
            test_name = 'Mann-Whitney U'
            effect = rank_biserial(stat, len(a), len(b))
            effect_name = 'rank-biserial r'

        test_results.append({
            'comparison': label,
            'feature': feat,
            'test': test_name,
            'statistic': round(stat, 4),
            'p_value': round(p, 6),
            'significant (α=0.05)': p < ALPHA,
            'larger_population': larger_population(a, b, test_name),
            effect_name: round(effect, 4),
            'mean_2025': round(np.mean(a), 4),
            'mean_2026': round(np.mean(b), 4),
            'median_2025': round(np.median(a), 4),
            'median_2026': round(np.median(b), 4),
        })

results_df = pd.DataFrame(test_results)

## Summary — not significant differences

In [ ]:
sig = results_df[results_df['significant (α=0.05)'] == False]
print(f'Not significant results: {len(sig)} / {len(results_df)}')

available_effect_cols = [c for c in ["Cohen's d", 'rank-biserial r'] if c in results_df.columns]
sig[['comparison', 'feature', 'test', 'p_value', 'significant (α=0.05)', 'larger_population'] + available_effect_cols + ['mean_2025', 'mean_2026', 'median_2025', 'median_2026']]

## Summary — significant differences

In [ ]:
sig = results_df[results_df['significant (α=0.05)']]
print(f'Significant results: {len(sig)} / {len(results_df)}')

available_effect_cols = [c for c in ["Cohen's d", 'rank-biserial r'] if c in results_df.columns]
sig[['comparison', 'feature', 'test', 'p_value', 'significant (α=0.05)', 'larger_population'] + available_effect_cols + ['mean_2025', 'mean_2026', 'median_2025', 'median_2026']]

## Questions to answer about gameplay:
- Have the rules for obtaining vision scores changed? This may affect visionScore and visionScorePerMinute
- Have the cooldown of stealth wards' trinkets changed? This may affect wardsPlaced, wardsKilled, stealthWardsPlaced